# Building a Blogpost Writing Multi-Agent System with AutoGen + Groq


### What we build

```text
Topic
  |
  v
Writer -> First Draft
  |
  +--> Critic
  |
  +--> SEO Reviewer
  |
  +--> Legal Reviewer
  |
  +--> Ethics Reviewer
  |
  v
Meta Reviewer -> Consolidated Feedback
  |
  v
Writer -> Final Blog
  |
  v
Python Word-Count Validation
```

## 1. Install current AutoGen packages

In [ ]:
!pip install -q -U "autogen-agentchat" "autogen-ext[openai]"

In [ ]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version)
print("autogen-agentchat:", metadata.version("autogen-agentchat"))
print("autogen-core:", metadata.version("autogen-core"))
print("autogen-ext:", metadata.version("autogen-ext"))

## 2. Configure the Groq API key

In [ ]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it in Colab Secrets and enable access."
    )

print("GROQ_API_KEY loaded successfully.")

## 3. Create the Groq model client

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
        "family": "unknown",
    },
    # Groq does not require the OpenAI `name` field.
    include_name_in_message=False,
)

print("Model client created.")

## 4. Test the connection before building the workflow

This is an important troubleshooting pattern:

```text
Test model
   |
   +-- works --> build agents
   |
   +-- fails --> fix API/environment first
```

In [ ]:
from autogen_agentchat.agents import AssistantAgent

test_agent = AssistantAgent(
    name="test_agent",
    model_client=model_client,
    system_message="You are a helpful assistant. Answer briefly.",
)

test_result = await test_agent.run(
    task="In one sentence, explain what a multi-agent system is."
)

print(test_result.messages[-1].content)

# 5. Define the blog task

In [ ]:
TASK = """
Write a blog post titled:

How Google Cloud AI is Helping Businesses and Developers

Requirements:
- Maximum 300 words.
- Include a clear title.
- Make it engaging and useful.
- Write for a general technology/business audience.
- Avoid unsupported statistics and exaggerated claims.
"""

print(TASK)

# 6. Create the Writer Agent

The Writer specializes in producing and revising content.

The **system message defines the agent's role**.

In [ ]:
writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="""
You are a professional technology writer.

Write concise, engaging blog posts.
Follow the user's requirements carefully.
Do not invent statistics, citations, customer names, or factual claims.
When reviewer feedback is provided, revise the article accordingly.

Return only the article, without commentary about your process.
""",
)

# 7. Generate the first draft

In [ ]:
draft_result = await writer.run(task=TASK)
draft = draft_result.messages[-1].content

print(draft)

# 8. Create the Critic Agent

The Critic does **not** rewrite the article.

It identifies problems with:

- clarity
- structure
- accuracy
- repetition
- audience fit
- unsupported claims

In [ ]:
critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    system_message="""
You are a senior editor reviewing a technology blog post.

Analyze the draft for clarity, structure, accuracy, usefulness,
engagement, repetition, audience fit, and unsupported claims.

Give concise, actionable feedback.
Do not rewrite the complete article.
""",
)

# 9. Review the draft

In [ ]:
critic_result = await critic.run(
    task=f"""
Review the following draft:

---------------- DRAFT ----------------
{draft}
-------------- END DRAFT --------------

Give concise, actionable editorial feedback.
"""
)

critic_feedback = critic_result.messages[-1].content

print(critic_feedback)

# 10. Create specialist reviewers

Instead of one reviewer checking everything, we use specialists.

```text
Draft
 |
 +--> SEO Reviewer
 |
 +--> Legal Reviewer
 |
 +--> Ethics Reviewer
```

Their work is independent, so it can be executed concurrently.

In [ ]:
seo_reviewer = AssistantAgent(
    name="seo_reviewer",
    model_client=model_client,
    system_message="""
You are an SEO and content-quality reviewer.

Check title quality, topic relevance, structure, readability,
search intent, and unnecessary repetition.

Return no more than 5 concise recommendations.
""",
)

legal_reviewer = AssistantAgent(
    name="legal_reviewer",
    model_client=model_client,
    system_message="""
You are a legal-risk reviewer for technology content.

Look for absolute or guaranteed claims, unsupported claims about
companies/products, misleading wording, and statements that should
be qualified.

Do not provide legal advice. Give editorial recommendations only.
""",
)

ethics_reviewer = AssistantAgent(
    name="ethics_reviewer",
    model_client=model_client,
    system_message="""
You are an ethics and responsible-AI reviewer.

Check for misleading framing, unfair comparisons, irresponsible AI
claims, hidden assumptions, bias, or potentially harmful messaging.

Return concise, actionable recommendations.
""",
)

# 11. Run the three specialist reviewers

Because the reviews are independent, we use `asyncio.gather()`.

This is an example of **parallel multi-agent work**.

In [ ]:
import asyncio

review_prompt = f"""
Review this blog post:

---------------- DRAFT ----------------
{draft}
-------------- END DRAFT --------------
"""

seo_task = seo_reviewer.run(task=review_prompt)
legal_task = legal_reviewer.run(task=review_prompt)
ethics_task = ethics_reviewer.run(task=review_prompt)

seo_result, legal_result, ethics_result = await asyncio.gather(
    seo_task,
    legal_task,
    ethics_task,
)

seo_feedback = seo_result.messages[-1].content
legal_feedback = legal_result.messages[-1].content
ethics_feedback = ethics_result.messages[-1].content

print("=== SEO REVIEW ===")
print(seo_feedback)

print("\n=== LEGAL REVIEW ===")
print(legal_feedback)

print("\n=== ETHICS REVIEW ===")
print(ethics_feedback)

# 12. Create the Meta Reviewer

The Meta Reviewer combines all specialist feedback into one prioritized revision plan.

It does not write the blog.

In [ ]:
meta_reviewer = AssistantAgent(
    name="meta_reviewer",
    model_client=model_client,
    system_message="""
You are the lead editor coordinating several specialist reviewers.

Combine their feedback into one prioritized revision plan.
Resolve duplicate recommendations.
Prioritize correctness, clarity, and usefulness.
Do not invent facts.
Keep the plan concise.
""",
)

In [ ]:
meta_prompt = f"""
Original blog draft:

---------------- DRAFT ----------------
{draft}
-------------- END DRAFT --------------

GENERAL CRITIC:
{critic_feedback}

SEO REVIEW:
{seo_feedback}

LEGAL REVIEW:
{legal_feedback}

ETHICS REVIEW:
{ethics_feedback}

Create a consolidated revision plan for the Writer.
"""

meta_result = await meta_reviewer.run(task=meta_prompt)
meta_feedback = meta_result.messages[-1].content

print(meta_feedback)

# 13. Writer produces the final version

Now the Writer receives the original task plus all the review information.

This is the final collaboration step.

In [ ]:
final_prompt = f"""
Original task:
{TASK}

Original draft:
{draft}

General critic feedback:
{critic_feedback}

SEO feedback:
{seo_feedback}

Legal feedback:
{legal_feedback}

Ethics feedback:
{ethics_feedback}

Consolidated editor feedback:
{meta_feedback}

Rewrite the article.

Requirements:
- Follow the original task.
- Apply useful reviewer feedback.
- Do not invent facts.
- Keep the article within 650 words.
- Return ONLY the final blog post.
"""

final_result = await writer.run(task=final_prompt)
final_blog = final_result.messages[-1].content

print(final_blog)

In [ ]:
len(final_prompt)

# 14. Validate the final article with Python

This demonstrates an important agent-design principle:

> Use an LLM for reasoning and writing, but use deterministic Python for deterministic checks.

In [ ]:
import re

word_count = len(re.findall(r"\b\w+[\w'-]*\b", final_blog))

print("Final word count:", word_count)

if word_count <= 650:
    print("✅ Word-count requirement satisfied.")
else:
    print("⚠️ Word-count requirement exceeded.")

# 15. Final architecture

```text
                         USER TASK
                             |
                             v
                        +---------+
                        |  Writer |
                        +---------+
                             |
                         First Draft
                             |
          +------------------+------------------+
          |                  |                  |
          v                  v                  v
       Critic             SEO Agent        Legal Agent
                                               |
                                               |
                                           Ethics Agent
          |                  |                  |
          +------------------+------------------+
                             |
                             v
                      Meta Reviewer
                             |
                      Revision Plan
                             |
                             v
                         Writer
                             |
                             v
                       FINAL BLOG
                             |
                             v
                    Python Validation
```

### What makes this a multi-agent system?

It is not merely multiple LLM calls.

The important pattern is:

```text
Specialized roles
      +
Independent outputs
      +
Orchestration
      +
Shared task context
      =
Multi-agent workflow
```

In [ ]:
# Cleanup: close the model client when you are finished.
await model_client.close()
print("Model client closed.")